In [60]:
import os
print(os.listdir())

['.config', '.ipynb_checkpoints', 'twitter_input.txt', 'output.json', 'sample_data']


In [61]:
import re
import uuid
import unicodedata
from datetime import datetime
from difflib import SequenceMatcher


# 🔹 LIMPIEZA DE TEXTO
def clean_text(text):
    text = text.lower()
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# 🔹 EXTRAER CÉDULA
def extract_cedula(text):
    patrones = [
        r'ci\s*\d{6,8}',
        r'cedula\s*\d{6,8}',
        r'identidad\s*\d{6,8}',
        r'documento\s*\d{6,8}',
        r'\b\d{6,8}\b'
    ]

    for patron in patrones:
        match = re.search(patron, text.lower())
        if match:
            numero = re.search(r'\d{6,8}', match.group())
            if numero:
                return numero.group()

    return None


#  FUZZY BASE
def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()


#  EXTRACT NAME (REEMPLAZA EL TUYO)
def extract_name(text):

    text_lower = text.lower()

    lugares = [
        "caracas", "chacao", "altamira", "petare", "catia",
        "la guaira", "valencia", "maracay", "barquisimeto",
        "maracaibo", "merida", "barinas"
    ]

    basura = [
        "en", "la", "el", "los", "las", "de", "del",
        "no", "persona", "extraviada", "extraviado",
        "perdida", "perdido", "desaparecida", "desaparecido",
        "localizada", "ayuda", "urgente", "rescate", "familia"
    ]

    patterns = [
        r'buscan a ([a-z]+ [a-z]+)',
        r'desaparecido ([a-z]+ [a-z]+)',
        r'desaparecida ([a-z]+ [a-z]+)',
        r'no sabemos nada de ([a-z]+ [a-z]+)',
        r'persona desaparecida ([a-z]+ [a-z]+)',
        r'persona extraviada ([a-z]+ [a-z]+)',
        r'([a-z]+ [a-z]+) ci'
    ]

    for pattern in patterns:
        match = re.search(pattern, text_lower)

        if match:
            nombre = match.group(1).strip()

            palabras = nombre.split()

            if len(palabras) != 2:
                continue

            if any(word in basura for word in palabras):
                continue

            if any(lugar in nombre for lugar in lugares):
                continue

            return nombre.title()

    return None


# 🔹 CATEGORIZAR
def categorize(text):
    text = text.lower()

    # 🔴 RESCATE (prioridad máxima)
    if any(x in text for x in [
        "rescate", "auxilio", "atrapado", "colapsado",
        "derrumb", "inundado", "no pueden salir",
        "edificio caido", "zona inundada"
    ]):
        return "necesita rescate"

    # 🔵 HOSPITAL
    if any(x in text for x in [
        "hospital", "clinica", "herido", "lesionado",
        "ingresado", "emergencia", "quirófano"
    ]):
        return "hospitalizado"

    # 🟡 DESAPARECIDO (AMPLIADO)
    if any(x in text for x in [
        "desaparecido", "desaparecida",
        "no aparece", "no sabemos de",
        "no sabemos nada", "sin contacto",
        "extraviado", "extraviada",
        "perdido", "perdida",
        "no responde", "se busca",
        "buscan a", "buscando a",
        "se busca a", "persona desaparecida",
        "persona extraviada",
        "persona perdida",
        "no localizado", "no localizada",
        "paradero desconocido"
    ]):
        return "desaparecido"

    # 🟢 ENCONTRADO
    if any(x in text for x in [
        "localizado", "encontrado", "aparecio",
        "ya aparecio", "persona localizada"
    ]):
        return "encontrado"

    # 🟣 CENTRO DE AYUDA
    if any(x in text for x in [
        "centro de acopio", "centro de ayuda",
        "donaciones", "recoleccion",
        "voluntarios", "punto de ayuda",
        "recibiendo ayuda"
    ]):
        return "centro de ayuda"

    # 🟠 NECESITA AYUDA (AMPLIADO)
    if any(x in text for x in [
        "necesitamos ayuda", "necesitan ayuda",
        "ayuda urgente", "ayuda requerida",
        "urgente ayuda",
        "sin comida", "sin agua", "sin medicinas",
        "no tenemos comida", "no tenemos agua",
        "falta comida", "falta agua",
        "hambre", "necesidades",
        "ayuda humanitaria"
    ]):
        return "necesita ayuda"

    # 🟢 INFRAESTRUCTURA (DAÑOS)
    if any(x in text for x in [
        "daños", "daño", "estructura dañada",
        "edificio afectado", "casa destruida",
        "vivienda afectada", "infraestructura",
        "zona afectada", "edificio con daños"
    ]):
        return "infraestructura"

    # ⚫ FALLECIDOS
    if any(x in text for x in [
        "fallecido", "murio", "murieron",
        "muerto", "sin vida", "cadaver"
    ]):
        return "fallecido"

    # 🐾 MASCOTAS
    if any(x in text for x in [
        "perro", "gato", "mascota", "animal"
    ]):
        return "mascotas"

    # 🔴 DENUNCIA
    if any(x in text for x in [
        "denuncia", "corrupcion", "robo",
        "saqueo", "fraude"
    ]):
        return "denuncia"

    return "otro"

# 🔹 UBICACIÓN
def extract_location(text):
    lugares = [
        "caracas", "chacao", "altamira", "petare",
        "la guaira", "valencia", "maracay",
        "barquisimeto", "maracaibo", "merida", "barinas"
    ]

    text_lower = text.lower()

    for lugar in lugares:
        if lugar in text_lower:
            return lugar.title()

    return None


# 🔹 ID
def generate_case_id(text, tipo, ubicacion):
    base = f"{tipo}_{ubicacion}_{text.lower()}"
    return str(uuid.uuid5(uuid.NAMESPACE_DNS, base))


# 🔹 LIMPIEZA FINAL
def remove_illegal_characters(text):
    if text:
        return re.sub(r'[\x00-\x1F\x7F]', '', text)
    return text

In [62]:
import os

files = [
    "twitter_input.txt",
    "whatsapp_input.txt",
    "facebook_input.txt",
    "instagram_input.txt",
    "linkedin_input.txt",
    "reddit_input.txt",
    "youtube_input.txt",
    "tiktok_input.txt",
    "telegram_input.txt"
]

raw_data = []

for file_name in files:
    print(f"Verificando: {file_name}")

    if os.path.exists(file_name):
        print(f" Encontrado: {file_name}")

        with open(file_name, "r", encoding="utf-16", errors="ignore") as f:
            lines = f.readlines()
            lines = [line.strip() for line in lines if line.strip()]
            raw_data.extend(lines)
    else:
        print(f" No existe: {file_name}")

print("TOTAL MENSAJES:", len(raw_data))
raw_data[:5]

Verificando: twitter_input.txt
 Encontrado: twitter_input.txt
Verificando: whatsapp_input.txt
 No existe: whatsapp_input.txt
Verificando: facebook_input.txt
 No existe: facebook_input.txt
Verificando: instagram_input.txt
 No existe: instagram_input.txt
Verificando: linkedin_input.txt
 No existe: linkedin_input.txt
Verificando: reddit_input.txt
 No existe: reddit_input.txt
Verificando: youtube_input.txt
 No existe: youtube_input.txt
Verificando: tiktok_input.txt
 No existe: tiktok_input.txt
Verificando: telegram_input.txt
 No existe: telegram_input.txt
TOTAL MENSAJES: 40


['AYUDA URGENTE familia atrapada en valencia edificio colapsado',
 'buscan a jose perez CI 12345678 en caracas desaparecido',
 'centro de acopio en chacao agua comida ropa',
 'rescate urgente en la guaira edificio derrumbado',
 'no aparece maria lopez desde ayer en maracay']

In [63]:
clean_data = []

for text in raw_data:

    text_clean = clean_text(text)

    cedula = extract_cedula(text_clean)
    tipo = categorize(text_clean)
    ubicacion = extract_location(text_clean)

    if isinstance(ubicacion, str):
        ubicacion = ubicacion.strip()

    nombre = extract_name(text)

    if nombre:
        nombre = nombre.replace("Desaparecido", "") \
                       .replace("Desaparecida", "") \
                       .strip()

    if nombre and nombre.lower() in ["la guaira", "caracas", "valencia"]:
        nombre = None

    record = {
        "id_caso": generate_case_id(text_clean, tipo, ubicacion),
        "cedula": cedula,
        "tipo": tipo,
        "nombre": nombre,
        "ubicacion": ubicacion,
        "fecha_registro": datetime.now().strftime("%Y-%m-%d"),
        "texto_original": remove_illegal_characters(text)
    }

    clean_data.append(record)

In [64]:
print("TOTAL RAW:", len(raw_data))
print("TOTAL CLEAN:", len(clean_data))
print(clean_data[:3])

TOTAL RAW: 40
TOTAL CLEAN: 40
[{'id_caso': '950f8a5e-91f6-5dee-a899-e82ef3a2485e', 'cedula': None, 'tipo': 'necesita rescate', 'nombre': None, 'ubicacion': 'Valencia', 'fecha_registro': '2026-06-30', 'texto_original': 'AYUDA URGENTE familia atrapada en valencia edificio colapsado'}, {'id_caso': 'a6e3e5d9-3da3-5f2e-96fe-a7e0e7e7e175', 'cedula': '12345678', 'tipo': 'desaparecido', 'nombre': 'Jose Perez', 'ubicacion': 'Caracas', 'fecha_registro': '2026-06-30', 'texto_original': 'buscan a jose perez CI 12345678 en caracas desaparecido'}, {'id_caso': 'cd14dc19-b72a-5f8d-9598-28aa3c3870ef', 'cedula': None, 'tipo': 'centro de ayuda', 'nombre': None, 'ubicacion': 'Chacao', 'fecha_registro': '2026-06-30', 'texto_original': 'centro de acopio en chacao agua comida ropa'}]


In [65]:
deduplicated = {}

for item in clean_data:
    if item["cedula"]:
        key = item["cedula"]
    else:
        key = item["id_caso"]

    deduplicated[key] = item

clean_data = list(deduplicated.values())


import json

json_output = json.dumps(clean_data, indent=2, ensure_ascii=False)
print(json_output)

[
  {
    "id_caso": "950f8a5e-91f6-5dee-a899-e82ef3a2485e",
    "cedula": null,
    "tipo": "necesita rescate",
    "nombre": null,
    "ubicacion": "Valencia",
    "fecha_registro": "2026-06-30",
    "texto_original": "AYUDA URGENTE familia atrapada en valencia edificio colapsado"
  },
  {
    "id_caso": "a6e3e5d9-3da3-5f2e-96fe-a7e0e7e7e175",
    "cedula": "12345678",
    "tipo": "desaparecido",
    "nombre": "Jose Perez",
    "ubicacion": "Caracas",
    "fecha_registro": "2026-06-30",
    "texto_original": "buscan a jose perez CI 12345678 en caracas desaparecido"
  },
  {
    "id_caso": "cd14dc19-b72a-5f8d-9598-28aa3c3870ef",
    "cedula": null,
    "tipo": "centro de ayuda",
    "nombre": null,
    "ubicacion": "Chacao",
    "fecha_registro": "2026-06-30",
    "texto_original": "centro de acopio en chacao agua comida ropa"
  },
  {
    "id_caso": "ac819d64-5d40-5655-ae24-a1fc7b35c55c",
    "cedula": null,
    "tipo": "necesita rescate",
    "nombre": null,
    "ubicacion": "La Gua

In [66]:
with open("output.json", "w", encoding="utf-8") as f:
    f.write(json_output)

print("Archivo guardado")

Archivo guardado


In [67]:
import pandas as pd
from datetime import datetime

df = pd.DataFrame(clean_data)

fecha = datetime.now().strftime("%Y-%m-%d")
filename = f"output_{fecha}.xlsx"

df.to_excel(filename, index=False)

print(f"Archivo generado: {filename}")

Archivo generado: output_2026-06-30.xlsx


In [68]:
from google.colab import files
files.download(filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [69]:
print(len(raw_data))
print(raw_data[:3])

40
['AYUDA URGENTE familia atrapada en valencia edificio colapsado', 'buscan a jose perez CI 12345678 en caracas desaparecido', 'centro de acopio en chacao agua comida ropa']
